# Ladybug: query a Star Wars graph stored in DuckDB

This notebook keeps the Star Wars graph in a DuckDB database and queries it from Ladybug without copying the tables into native Ladybug storage.

The flow is: build ordinary DuckDB `characters` and `interactions` tables -> attach the DuckDB database from Ladybug -> declare a Ladybug relationship table backed by the attached DuckDB relationship table -> run Cypher over the attached DuckDB graph.

## Environment

Dependencies are declared in `pyproject.toml`. From a fresh checkout, start Jupyter with:

```bash
uv run jupyter notebook ladybug_duckdb_attached_starwars.ipynb
```

Or execute the notebook from the CLI with:

```bash
uv run jupyter nbconvert --to notebook --execute ladybug_duckdb_attached_starwars.ipynb --inplace
```

In [ ]:
# Optional setup cell: sync dependencies from pyproject.toml.
# If you launched with `uv run jupyter notebook`, this is already done.
!uv sync

In [ ]:
from pathlib import Path
import shutil

import duckdb
import ladybug
import pyarrow as pa

print("ladybug", ladybug.__version__)
print("duckdb", duckdb.__version__)

## Build the same Star Wars graph tables

The node table has one row per character. The relationship table stores endpoint IDs in its first two columns, `source` and `target`, whose values are the `Character.id` primary keys.

In [ ]:
characters = pa.table({
    "id": pa.array(list(range(18)), type=pa.int64()),
    "name": [
        "Luke Skywalker", "Leia Organa", "Han Solo", "Chewbacca", "R2-D2", "C-3PO",
        "Obi-Wan Kenobi", "Yoda", "Darth Vader", "Emperor Palpatine", "Tarkin",
        "Boba Fett", "Lando Calrissian", "Jabba the Hutt", "Mon Mothma",
        "Wedge Antilles", "Mara Jade", "Ahsoka Tano",
    ],
    "faction": [
        "Rebellion", "Rebellion", "Rebellion", "Rebellion", "Rebellion", "Rebellion",
        "Jedi", "Jedi", "Empire", "Empire", "Empire", "Underworld", "Rebellion",
        "Underworld", "Rebellion", "Rebellion", "Underworld", "Jedi",
    ],
    "homeworld": [
        "Tatooine", "Alderaan", "Corellia", "Kashyyyk", "Naboo", "Tatooine",
        "Stewjon", "Dagobah", "Tatooine", "Naboo", "Eriadu", "Kamino", "Socorro",
        "Nal Hutta", "Chandrila", "Corellia", "Coruscant", "Shili",
    ],
    "role": [
        "Jedi hopeful", "General", "Smuggler", "Co-pilot", "Astromech", "Protocol droid",
        "Mentor", "Grand master", "Sith lord", "Emperor", "Grand Moff", "Bounty hunter",
        "Administrator", "Crime lord", "Chancellor", "Pilot", "Agent", "Fulcrum",
    ],
})

name_to_id = dict(zip(characters["name"].to_pylist(), characters["id"].to_pylist()))

interaction_names = pa.table({
    "source": [
        "Luke Skywalker", "Luke Skywalker", "Han Solo", "Leia Organa", "R2-D2", "R2-D2",
        "C-3PO", "Obi-Wan Kenobi", "Yoda", "Obi-Wan Kenobi", "Ahsoka Tano", "Ahsoka Tano",
        "Darth Vader", "Darth Vader", "Tarkin", "Darth Vader", "Darth Vader", "Darth Vader",
        "Boba Fett", "Boba Fett", "Jabba the Hutt", "Jabba the Hutt", "Lando Calrissian",
        "Lando Calrissian", "Mon Mothma", "Mon Mothma", "Wedge Antilles", "Wedge Antilles",
        "Mara Jade", "Mara Jade", "Mara Jade",
    ],
    "target": [
        "Leia Organa", "Han Solo", "Chewbacca", "Han Solo", "C-3PO", "Luke Skywalker",
        "Leia Organa", "Luke Skywalker", "Luke Skywalker", "Yoda", "Obi-Wan Kenobi", "Yoda",
        "Emperor Palpatine", "Tarkin", "Emperor Palpatine", "Luke Skywalker", "Leia Organa",
        "Obi-Wan Kenobi", "Darth Vader", "Jabba the Hutt", "Han Solo", "Leia Organa",
        "Han Solo", "Leia Organa", "Leia Organa", "Wedge Antilles", "Luke Skywalker",
        "Lando Calrissian", "Emperor Palpatine", "Luke Skywalker", "Boba Fett",
    ],
    "scene": [
        "twins and rebellion", "rescues and raids", "lifelong crew", "command and chemistry",
        "droid duo", "secret plans", "translation and diplomacy", "training", "training",
        "jedi council", "clone wars", "jedi order", "master and apprentice", "imperial command",
        "death star politics", "family conflict", "imperial pursuit", "old master", "bounty contract",
        "underworld contract", "debt", "palace rescue", "old friends", "alliance planning",
        "rebel leadership", "fleet command", "rogue squadron", "battle of endor", "emperor's hand",
        "rivalry to trust", "underworld overlap",
    ],
    "strength": pa.array([8, 7, 10, 8, 9, 7, 5, 9, 9, 6, 6, 5, 10, 7, 6, 10, 6, 9, 6, 8, 8, 4, 8, 6, 8, 5, 7, 6, 7, 6, 4], type=pa.int64()),
})

interactions = pa.table({
    "source": pa.array([name_to_id[name] for name in interaction_names["source"].to_pylist()], type=pa.int64()),
    "target": pa.array([name_to_id[name] for name in interaction_names["target"].to_pylist()], type=pa.int64()),
    "scene": interaction_names["scene"],
    "strength": interaction_names["strength"],
})

characters, interactions

## Store the graph in DuckDB

This creates a normal DuckDB database. Ladybug will read the same `characters` and `interactions` tables through an attachment in the next section.

In [ ]:
work_dir = Path("/tmp/ladybug_attached_duckdb_starwars")
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True)

duckdb_path = work_dir / "starwars.duckdb"

with duckdb.connect(str(duckdb_path)) as duck:
    duck.register("characters_arrow", characters)
    duck.register("interactions_arrow", interactions)
    duck.execute("CREATE TABLE characters AS SELECT * FROM characters_arrow")
    duck.execute("CREATE TABLE interactions AS SELECT * FROM interactions_arrow")

    print("DuckDB path:", duckdb_path)
    print("characters:", duck.execute("SELECT count(*) FROM characters").fetchone()[0])
    print("interactions:", duck.execute("SELECT count(*) FROM interactions").fetchone()[0])
    display(duck.execute("SELECT * FROM characters ORDER BY id LIMIT 5").df())
    display(duck.execute("SELECT * FROM interactions ORDER BY strength DESC, scene LIMIT 5").df())

## Attach DuckDB from Ladybug

The `duckdb` extension lets Ladybug scan tables from an attached DuckDB database. The `LOAD FROM` query below is a quick inspection step before declaring graph tables.

In [ ]:
conn = ladybug.Connection(ladybug.Database())

# INSTALL is idempotent once the extension is present in the local Ladybug extension cache.
conn.execute("INSTALL duckdb")
conn.execute("LOAD duckdb")
conn.execute(f"ATTACH '{duckdb_path}' AS sw (dbtype duckdb)")

conn.query_as_arrow("""
LOAD FROM sw.characters
RETURN id, name, faction, homeworld, role
ORDER BY id
LIMIT 5
""", 4096).get_as_arrow().to_pandas()

## Declare a rel table backed by DuckDB

After `ATTACH`, the DuckDB node table is addressable directly as `sw.characters`. The only schema declaration we need is a Ladybug relationship table whose endpoints are `sw.characters` and whose storage is the attached DuckDB `sw.interactions` table.

In [ ]:
conn.execute("""
CREATE REL TABLE INTERACTS_WITH(
    FROM sw.characters TO sw.characters
)
WITH (storage = 'sw.interactions')
""")

## Inspect the pushed-down SQL

`EXPLAIN` includes the SQL that Ladybug pushes into the attached DuckDB database. This cell extracts just that SQL from the ASCII plan box.

In [ ]:
explain_df = conn.execute("""
EXPLAIN MATCH (a:sw.characters)-[r:INTERACTS_WITH]->(b:sw.characters)
RETURN a.name AS source, b.name AS target, r.scene AS scene, r.strength AS strength
ORDER BY r.strength DESC
LIMIT 5
""").get_as_df()

explain_text = explain_df.iloc[0, 0]
sql_parts = []
capturing = False

for line in explain_text.splitlines():
    if "Function:" in line:
        capturing = True
        line = line.split("Function:", 1)[1]
    elif capturing and ("Expressions:" in line or "NumOutputTuples:" in line):
        break

    if capturing:
        cleaned = line.replace("│", "").strip()
        if cleaned:
            sql_parts.append(cleaned)

pushed_down_sql = " ".join(sql_parts)
print(pushed_down_sql)

## Query the attached DuckDB graph with Cypher

The node label is the attached DuckDB table name, `sw.characters`. The relationship scan is backed by the attached DuckDB `sw.interactions` table.

In [ ]:
conn.query_as_arrow("""
MATCH (a:sw.characters)-[r:INTERACTS_WITH]->(b:sw.characters)
RETURN a.name AS source, a.faction AS source_faction,
       b.name AS target, b.faction AS target_faction,
       r.scene AS scene, r.strength AS strength
ORDER BY r.strength DESC
LIMIT 12
""", 4096).get_as_arrow().to_pandas()

In [ ]:
conn.query_as_arrow("""
MATCH (a:sw.characters)-[r:INTERACTS_WITH]->(b:sw.characters)
RETURN a.name AS character, a.faction AS faction,
       count(*) AS outgoing_interactions,
       sum(r.strength) AS total_strength
ORDER BY total_strength DESC, outgoing_interactions DESC, character
LIMIT 10
""", 4096).get_as_arrow().to_pandas()

In [ ]:
conn.query_as_arrow("""
MATCH (a:sw.characters)-[r:INTERACTS_WITH]->(b:sw.characters)
WHERE a.faction <> b.faction
RETURN a.faction AS source_faction, b.faction AS target_faction,
       count(*) AS interactions,
       sum(r.strength) AS total_strength
ORDER BY total_strength DESC, interactions DESC
""", 4096).get_as_arrow().to_pandas()

## What happened

1. The Star Wars graph was written as ordinary DuckDB tables.
2. Ladybug loaded the `duckdb` extension and attached that DuckDB database as `sw`.
3. `LOAD FROM sw.characters` verified that Ladybug can scan the attached node table.
4. `CREATE REL TABLE ... FROM sw.characters TO sw.characters WITH (storage='sw.interactions')` declared a relationship table backed by DuckDB.
5. Cypher `MATCH` queries traversed `sw.characters` through `INTERACTS_WITH` without copying those rows into native Ladybug storage.